In [15]:
pip install biopython pyfaidx pandas

Note: you may need to restart the kernel to use updated packages.


In [16]:
def get_sequence(chromosome, position, flank=8):
    """
    Extract a sequence of length (2*flank + 1) around the SNP position.

    Parameters:
    - chromosome: Chromosome name (e.g., 'chr1')
    - position: SNP position (1-based coordinate)
    - flank: Number of bases upstream and downstream of the SNP

    Returns:
    - sequence: DNA sequence as a string
    """
    # Adjust for 0-based indexing in pyfaidx
    start = position - flank - 1  # Subtract 1 because positions are 1-based
    end = position + flank        # End position is exclusive in slicing

    # Handle cases where start is negative
    if start < 0:
        start = 0

    # Extract the sequence
    try:
        seq = genome[chromosome][start:end]
        return str(seq).upper()
    except KeyError:
        print(f"Chromosome {chromosome} not found in the reference genome.")
        return None
    except Exception as e:
        print(f"Error retrieving sequence for {chromosome}:{position}: {e}")
        return None


# Define the padding function
def pad_sequence(seq, target_length=500):
    """
    Pads the input sequence with 'N's on both sides to reach the target length.
    The original sequence is centered. If the sequence is longer than target_length,
    it is truncated from the center.
    
    Parameters:
        seq (str): The original DNA sequence.
        target_length (int): The desired total length after padding.
    
    Returns:
        str: The padded (or truncated) sequence.
    """
    seq_length = len(seq)
    
    if seq_length >= target_length:
        # If the sequence is longer than or equal to target_length, truncate it
        # to the target_length by keeping the center part
        start = (seq_length - target_length) // 2
        end = start + target_length
        return seq[start:end]
    
    # Calculate total padding needed
    total_pad = target_length - seq_length
    
    # Calculate padding on each side
    pad_left = total_pad // 2
    pad_right = total_pad - pad_left
    
    # Create padded sequence
    padded_seq = 'N' * pad_left + seq + 'N' * pad_right
    return padded_seq

In [17]:
# Load SNP data
import pandas as pd
snp_data = pd.read_csv('indexing/SNP_haploregannotatr_ATAC_tss_SNPFunction_spliceai_bayesianML_Bellenguez_202401021.csv')

snp_data = snp_data[['RSID','chr','pos_hg38','Major','Minor','a1','a2']]#.drop_duplicates('RSID')
snp_data.columns = ['SNP_ID','chromosome','position','Major','Minor','a1','a2']

from pyfaidx import Fasta
from Bio.Seq import Seq
from Bio import SeqIO
import os
# Load the genome
genome = Fasta('/media/zihengc/T7/genome/hg38.fa' , as_raw=True)  # as_raw=True returns sequences as strings

# Create a new column for sequences
snp_data['Sequence'] = snp_data.apply(
    lambda row: get_sequence(row['chromosome'], row['position'], flank=8),
    axis=1
)

# Preview the updated DataFrame
print(snp_data.head())

missing_sequences = snp_data[snp_data['Sequence'].isnull()]
if not missing_sequences.empty:
    print("Sequences could not be retrieved for the following SNPs:")
    print(missing_sequences)

from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq

# Create SeqRecord objects
seq_records = []
for index, row in snp_data.iterrows():
    if pd.notnull(row['Sequence']):
        record = SeqRecord(
            Seq(row['Sequence']),
            id=row['SNP_ID'],
            description=f"{row['chromosome']}:{row['position']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('indexing/snp_bp_sequences.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

######################################################################
snp_data['hg38_center_base']=snp_data['Sequence'].str.slice(8,13)
snp_data['Major_len'] = snp_data['Major'].str.len()
snp_data['Minor_len'] = snp_data['Minor'].str.len()

# Create boolean mask
mask = (snp_data['Major'].str.len() == 1) & (snp_data['Minor'].str.len() == 1)
snp_data_single = snp_data[mask].copy()

# Create a boolean mask where 'Seq_13' matches 'Major' or 'Minor'
mask = snp_data_single.apply(
    lambda row: (row['hg38_center_base'] == row['Major']) or (row['hg38_center_base'] == row['Minor']),
    axis=1
)

# Filter the DataFrame
snp_data_single = snp_data_single[mask].copy()
snp_data_single['Sequence_Major'] = snp_data_single['Sequence'].str.slice(0,8)+snp_data_single['Major']+snp_data_single['Sequence'].str.slice(13,)
snp_data_single['Sequence_Minor'] = snp_data_single['Sequence'].str.slice(0,8)+snp_data_single['Minor']+snp_data_single['Sequence'].str.slice(13,)

snp_data_multi = snp_data.loc[~snp_data.index.isin(snp_data_single.index)]
#snp_data_multi.to_csv('snp_data_multi.csv')

######################mannually modified this file########################################

snp_data_multi_modified = pd.read_csv('allele_differences_motif_analysis/snp_data_multi_modified.csv',index_col=0)
snp_data_modified = pd.concat([snp_data_single,snp_data_multi_modified]).sort_index()

# Apply the padding function to 'Sequence_Major' and 'Sequence_Minor'
snp_data_modified['Sequence_Major_padded'] = snp_data_modified['Sequence_Major'].apply(pad_sequence)
snp_data_modified['Sequence_Minor_padded'] = snp_data_modified['Sequence_Minor'].apply(pad_sequence)

#snp_data_modified.to_csv("indexing/snp_data_modified.csv")

       SNP_ID chromosome  position Major Minor a1 a2           Sequence
0  cg03073402      chr19  42423524     C     G  C  G  CGTTCCCCCGGCCAACT
1  cg03169557      chr16  89532542     C     G  C  G  GATGAGATCGACGCGGT
2  cg05030077      chr16   2205198     C     G  C  G  GAGGGCACCGGAACAGC
3  cg05066959       chr8  41661790     C     G  C  G  GCCGCGGGCGCCTCAGT
4  cg05228284      chr19   2720849     C     G  C  G  GCACAGTGCGGGGCCAG


In [18]:

########################################################################
import pandas as pd
from pyfaidx import Fasta
from Bio.Seq import Seq
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
import os

# Ensure that the necessary directories exist
output_dir = 'indexing'
os.makedirs(output_dir, exist_ok=True)

# Load the genome
genome = Fasta('/media/zihengc/T7/genome/hg38.fa', as_raw=True)  # as_raw=True returns sequences as strings

def get_sequence(chromosome, position, flank=8):
    """
    Extract a sequence of length (2*flank + 1) around the SNP position.

    Parameters:
    - chromosome: Chromosome name (e.g., 'chr1')
    - position: SNP position (1-based coordinate)
    - flank: Number of bases upstream and downstream of the SNP

    Returns:
    - sequence: DNA sequence as a string
    """
    # Adjust for 0-based indexing in pyfaidx
    start = position - flank - 1  # Subtract 1 because positions are 1-based
    end = position + flank        # End position is exclusive in slicing

    # Handle cases where start is negative
    if start < 0:
        start = 0

    # Extract the sequence
    try:
        seq = genome[chromosome][start:end]
        return str(seq).upper()
    except KeyError:
        print(f"Chromosome {chromosome} not found in the reference genome.")
        return None
    except Exception as e:
        print(f"Error retrieving sequence for {chromosome}:{position}: {e}")
        return None

# Example: Loading your SNP data
# Replace this with your actual data loading method
# For demonstration, let's create a sample DataFrame
# Ensure your DataFrame has 'chromosome', 'position', and 'SNP_ID' columns


# Create a new column for sequences
snp_data['Sequence'] = snp_data.apply(
    lambda row: get_sequence(row['chromosome'], row['position'], flank=8),
    axis=1
)

# Add padding: 237 Ns to the left and 238 Ns to the right
padding_left = 'N' * 237
padding_right = 'N' * 238

def pad_sequence(seq):
    if seq:
        return f"{padding_left}{seq}{padding_right}"
    else:
        return None

snp_data['Padded_Sequence'] = snp_data['Sequence'].apply(pad_sequence)

# Verify that all padded sequences are 500 bp long
def verify_length(seq):
    if seq:
        return len(seq) == 237 +  + 238
    return False

snp_data['Is_Valid_Length'] = snp_data['Padded_Sequence'].apply(verify_length)

# Preview the updated DataFrame
print(snp_data.head())

# Identify any sequences that could not be retrieved or are improperly padded
missing_sequences = snp_data[snp_data['Padded_Sequence'].isnull()]
invalid_length_sequences = snp_data[~snp_data['Is_Valid_Length'] & snp_data['Padded_Sequence'].notnull()]

if not missing_sequences.empty:
    print("\nSequences could not be retrieved for the following SNPs:")
    print(missing_sequences[['SNP_ID', 'chromosome', 'position']])

if not invalid_length_sequences.empty:
    print("\nSequences with invalid lengths:")
    print(invalid_length_sequences[['SNP_ID', 'chromosome', 'position', 'Is_Valid_Length']])

# Create SeqRecord objects with padded sequences
seq_records = []
for index, row in snp_data.iterrows():
    if pd.notnull(row['Padded_Sequence']):
        record = SeqRecord(
            Seq(row['Padded_Sequence']),
            id=row['SNP_ID'],
            description=f"{row['chromosome']}:{row['position']}"
        )
        seq_records.append(record)

# Write to a FASTA file
fasta_output_path = os.path.join(output_dir, 'snp_bp_padded_500bp_sequences.fasta')
with open(fasta_output_path, 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

print(f"\nFASTA file with padded sequences has been written to: {fasta_output_path}")
#########################################################################################
#THP1 Macrophage
df_allele = pd.read_csv('/media/zihengc/T7/mpra3_lib_analysis/allele_differences_withoutcontrol/20240817_allele_only_for_plotting/annotated_20240813_comparative_THP1Macrophage_alleleOnly.csv',index_col=0)
df_allele['snp_bp_sequence'] = (padding_left+snp_data['Sequence']+padding_right).tolist()

# Create SeqRecord objects
seq_records = []
data_to_save = df_allele.drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['snp_bp_sequence']):
        record = SeqRecord(
            Seq(row['snp_bp_sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('THP1_macrophage_all_snps_bpsequence_500bpNs.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')


# Create SeqRecord objects
seq_records = []
data_to_save = df_allele[df_allele['fdr']<=0.05].drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['snp_bp_sequence']):
        record = SeqRecord(
            Seq(row['snp_bp_sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('THP1_macrophage_sig_snps_bpsequence_500bpNs.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')


# Create SeqRecord objects
seq_records = []
data_to_save = df_allele[df_allele['fdr']>0.05].drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['snp_bp_sequence']):
        record = SeqRecord(
            Seq(row['snp_bp_sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('THP1_macrophage_nonsig_snps_bpsequence_500bpNs.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

import pandas as pd
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO


# Define a function to create SeqRecord objects for both alleles
def create_seq_records(df, condition=None):
    """
    Creates SeqRecord objects for both Major and Minor alleles based on a condition.
    
    Parameters:
        df (pd.DataFrame): The DataFrame containing SNP data.
        condition (callable, optional): A function to filter the DataFrame rows.
        
    Returns:
        list: A list of SeqRecord objects.
    """
    seq_records = []
    
    # Apply the condition if provided
    if condition:
        data_to_save = df[condition].drop_duplicates('RSID')
    else:
        data_to_save = df.drop_duplicates('RSID')
    
    for index, row in data_to_save.iterrows():
        rsid = row['RSID']
        chr_ = row['chr']
        pos = row['pos_hg38']
        major_base = row['Major']
        minor_base = row['Minor']
        
        # Padded sequences
        seq_major_padded = row.get('Sequence_Major_padded', row.get('Sequence_Major'))
        seq_minor_padded = row.get('Sequence_Minor_padded', row.get('Sequence_Minor'))
        
        # Create SeqRecord for Major allele
        if pd.notnull(seq_major_padded):
            major_id = f"{rsid}_Major_{major_base}:#THP1_Macrophage"
            major_description = f"{chr_}:{pos}"
            record_major = SeqRecord(
                Seq(seq_major_padded),
                id=major_id,
                description=major_description
            )
            seq_records.append(record_major)
        
        # Create SeqRecord for Minor allele
        if pd.notnull(seq_minor_padded):
            minor_id = f"{rsid}_Minor_{minor_base}:#THP1_Macrophage"
            minor_description = f"{chr_}:{pos}"
            record_minor = SeqRecord(
                Seq(seq_minor_padded),
                id=minor_id,
                description=minor_description
            )
            seq_records.append(record_minor)
    
    return seq_records
##################################################################################

# Read and prepare the main SNP data with padded sequences
snp_data_multi_modified = pd.read_csv("indexing/snp_data_modified.csv",index_col=0)  # Update the path as needed

# ------------------------------
# a. Saving All SNPs
# ------------------------------
# Read the allele differences DataFrame
df_allele = pd.read_csv('/media/zihengc/T7/mpra3_lib_analysis/allele_differences_withoutcontrol/20240817_allele_only_for_plotting/annotated_20240813_comparative_THP1Macrophage_alleleOnly.csv', index_col=0)

df_allele = df_allele.join(
    snp_data_multi_modified.drop_duplicates('SNP_ID').set_index('SNP_ID')[
        ['Sequence_Major_padded', 'Sequence_Minor_padded']
    ],
    on='RSID',
    how='left'
)

# Create BED DataFrame for single base
data_to_save = df_allele[df_allele['fdr']<=0.05].drop_duplicates('RSID')
bed_single = pd.DataFrame({
    'chrom': data_to_save ['chr'],
    'chromStart': data_to_save['pos_hg38'] - 1,  # Convert to 0-based
    'chromEnd': data_to_save['pos_hg38'],        # 1-based, non-inclusive
    'name': data_to_save['RSID']})
bed_single.to_csv('THP1_macrophage_sig_snps.bed', sep='\t', header=False, index=False)

# Create BED DataFrame for single base
data_to_save = df_allele[df_allele['fdr']>0.05].drop_duplicates('RSID')
bed_single = pd.DataFrame({
    'chrom': data_to_save ['chr'],
    'chromStart': data_to_save['pos_hg38'] - 1,  # Convert to 0-based
    'chromEnd': data_to_save['pos_hg38'],        # 1-based, non-inclusive
    'name': data_to_save['RSID']})
bed_single.to_csv('THP1_macrophage_nonsig_snps.bed', sep='\t', header=False, index=False)

# Create SeqRecord objects for all SNPs
seq_records_all = create_seq_records(df_allele)


# Write to a FASTA file
with open('bp_all_snps_padded.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records_all, fasta_file, 'fasta')
# ------------------------------
# b. Saving Significant SNPs (fdr <= 0.05)
# ------------------------------
# Create SeqRecord objects for significant SNPs
seq_records_sig = create_seq_records(df_allele, condition=lambda df: df['fdr'] <= 0.05)

# Write to a FASTA file
with open('THP1_macrophage_sig_snps_bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records_sig, fasta_file, 'fasta')

# ------------------------------
# c. Saving Non-Significant SNPs (fdr > 0.05)
# ------------------------------
# Create SeqRecord objects for non-significant SNPs
seq_records_nonsig = create_seq_records(df_allele, condition=lambda df: df['fdr'] > 0.05)

# Write to a FASTA file
with open('THP1_macrophage_nonsig_snps_bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records_nonsig, fasta_file, 'fasta')

# ------------------------------
# d. Saving Causal Minor Alleles (Reducing Transcription)
# ------------------------------
# Read the causal minor alleles DataFrame (Reducing Transcription)
df_allele_reducing = pd.read_csv('/media/zihengc/T7/mpra3_lib_analysis/causal_alleles/20240819_THP1Macrophage_CausalMinorAllele_ReducingTranscription_combined_withMAD.csv', index_col=0).drop('SNP_ID', axis=1)

# Merge with snp_data to get Sequence_Major and Sequence_Minor
df_allele_reducing = df_allele_reducing.merge(snp_data_multi_modified[['SNP_ID', 'Sequence_Major_padded', 'Sequence_Minor_padded']], left_on='RSID', right_on='SNP_ID', how='left')

# Create SeqRecord objects for significant causal minor alleles (Reducing)
seq_records_causal_reducing = create_seq_records(df_allele_reducing, condition=lambda df: df['fdr'] <= 0.05)

# Write to a FASTA file
with open('THP1_macrophage_causal_minor_reducing_snps_bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records_causal_reducing, fasta_file, 'fasta')

# Create BED DataFrame for causal minor reducing SNPs
bed_causal_reducing = df_allele_reducing[df_allele_reducing['fdr'] <= 0.05].drop_duplicates('RSID')
bed_causal_reducing_bed = pd.DataFrame({
    'chrom': bed_causal_reducing['chr'],
    'chromStart': (bed_causal_reducing['pos_hg38'] - 1).astype(int),  # Convert to 0-based
    'chromEnd': bed_causal_reducing['pos_hg38'].astype(int),         # 1-based, non-inclusive
    'name': bed_causal_reducing['RSID']
})

# Write to a BED file
bed_causal_reducing_bed.to_csv('THP1_macrophage_causal_minor_reducing_snps.bed', sep='\t', header=False, index=False)

# ------------------------------
# e. Saving Causal Minor Alleles (Increasing Transcription)
# ------------------------------
# Read the causal minor alleles DataFrame (Increasing Transcription)
df_allele_increasing = pd.read_csv('/media/zihengc/T7/mpra3_lib_analysis/causal_alleles/20240819_THP1Macrophage_CausalMinorAllele_IncreasingTranscription_combined_withMAD.csv', index_col=0).drop('SNP_ID', axis=1)

# Merge with snp_data to get Sequence_Major and Sequence_Minor
df_allele_increasing = df_allele_increasing.merge(snp_data_multi_modified[['SNP_ID', 'Sequence_Major_padded', 'Sequence_Minor_padded']], left_on='RSID', right_on='SNP_ID', how='left')

# Create SeqRecord objects for significant causal minor alleles (Increasing)
seq_records_causal_increasing = create_seq_records(df_allele_increasing, condition=lambda df: df['fdr'] <= 0.05)

# Write to a FASTA file
with open('THP1_macrophage_causal_minor_increasing_snps_bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records_causal_increasing, fasta_file, 'fasta')

# Create BED DataFrame for causal minor increasing SNPs
bed_causal_increasing = df_allele_increasing[df_allele_increasing['fdr'] <= 0.05].drop_duplicates('RSID')
bed_causal_increasing_bed = pd.DataFrame({
    'chrom': bed_causal_increasing['chr'],
    'chromStart': (bed_causal_increasing['pos_hg38'] - 1).astype(int),  # Convert to 0-based
    'chromEnd': bed_causal_increasing['pos_hg38'].astype(int),         # 1-based, non-inclusive
    'name': bed_causal_increasing['RSID']
})

# Write to a BED file
bed_causal_increasing_bed.to_csv('THP1_macrophage_causal_minor_increasing_snps.bed', sep='\t', header=False, index=False)

       SNP_ID chromosome  position Major Minor a1 a2           Sequence  \
0  cg03073402      chr19  42423524     C     G  C  G  CGTTCCCCCGGCCAACT   
1  cg03169557      chr16  89532542     C     G  C  G  GATGAGATCGACGCGGT   
2  cg05030077      chr16   2205198     C     G  C  G  GAGGGCACCGGAACAGC   
3  cg05066959       chr8  41661790     C     G  C  G  GCCGCGGGCGCCTCAGT   
4  cg05228284      chr19   2720849     C     G  C  G  GCACAGTGCGGGGCCAG   

  hg38_center_base  Major_len  Minor_len  \
0            CGGCC          1          1   
1            CGACG          1          1   
2            CGGAA          1          1   
3            CGCCT          1          1   
4            CGGGG          1          1   

                                     Padded_Sequence  Is_Valid_Length  
0  NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...            False  
1  NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...            False  
2  NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...            Fals

In [19]:
#Mouse Brain
df_allele = pd.read_csv('/media/zihengc/T7/mpra3_lib_analysis/allele_differences_withoutcontrol/20240817_allele_only_for_plotting/annotated_20240616_comparative_BrainR1R2merged20240404_alleleOnly.csv',index_col=0)
df_allele['snp_bp_sequence'] = snp_data['Sequence'].tolist()

# Create SeqRecord objects
seq_records = []
data_to_save = df_allele.drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['snp_bp_sequence']):
        record = SeqRecord(
            Seq(row['snp_bp_sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('mouse_brain_all_snps_bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')


# Create SeqRecord objects
seq_records = []
data_to_save = df_allele[df_allele['fdr']<=0.05].drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['snp_bp_sequence']):
        record = SeqRecord(
            Seq(row['snp_bp_sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('mouse_brain_sig_snps_bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')


# Create SeqRecord objects
seq_records = []
data_to_save = df_allele[df_allele['fdr']>0.05].drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['snp_bp_sequence']):
        record = SeqRecord(
            Seq(row['snp_bp_sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('mouse_brain_nonsig_snps_bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

# Create BED DataFrame for single base
data_to_save = df_allele.drop_duplicates('RSID')
bed_single = pd.DataFrame({
    'chrom': data_to_save ['chr'],
    'chromStart': data_to_save['pos_hg38'] - 1,  # Convert to 0-based
    'chromEnd': data_to_save['pos_hg38'],        # 1-based, non-inclusive
    'name': data_to_save['RSID']
})


bed_single.to_csv('mouse_brain_all_snps.bed', sep='\t', header=False, index=False)

# Create BED DataFrame for single base
data_to_save = df_allele[df_allele['fdr']<=0.05].drop_duplicates('RSID')
bed_single = pd.DataFrame({
    'chrom': data_to_save ['chr'],
    'chromStart': data_to_save['pos_hg38'] - 1,  # Convert to 0-based
    'chromEnd': data_to_save['pos_hg38'],        # 1-based, non-inclusive
    'name': data_to_save['RSID']
})


bed_single.to_csv('mouse_brain_sig_snps.bed', sep='\t', header=False, index=False)


df_allele = pd.read_csv('/media/zihengc/T7/mpra3_lib_analysis/causal_alleles/20240819_BrainR1R2merged20240404_CausalMinorAllele_ReducingTranscription_combined_withMAD.csv' ,index_col=0).drop('SNP_ID',axis=1)
df_allele = pd.merge(df_allele,snp_data,left_on='RSID',right_on='SNP_ID').drop_duplicates('RSID')

# Create SeqRecord objects
seq_records = []
data_to_save = df_allele[df_allele['fdr']<=0.05].drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['Sequence']):
        record = SeqRecord(
            Seq(row['Sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('mouse_brain_causal_minor_reducing_snps_bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

# Create BED DataFrame for single base
data_to_save = df_allele.drop_duplicates('RSID')
bed_single = pd.DataFrame({
    'chrom': data_to_save ['chr'],
    'chromStart': (data_to_save['pos_hg38'] - 1).astype(int),  # Convert to 0-based
    'chromEnd': data_to_save['pos_hg38'].astype(int),        # 1-based, non-inclusive
    'name': data_to_save['RSID']
})

bed_single.to_csv('mouse_brain_causal_minor_reducing_snps.bed', sep='\t', header=False, index=False)



df_allele = pd.read_csv('/media/zihengc/T7/mpra3_lib_analysis/causal_alleles/20240819_BrainR1R2merged20240404_CausalMinorAllele_increasingTranscription_combined_withMAD.csv' ,index_col=0).drop('SNP_ID',axis=1)
df_allele = pd.merge(df_allele,snp_data,left_on='RSID',right_on='SNP_ID').drop_duplicates('RSID')

# Create SeqRecord objects
seq_records = []
data_to_save = df_allele[df_allele['fdr']<=0.05].drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['Sequence']):
        record = SeqRecord(
            Seq(row['Sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('mouse_brain_causal_minor_reducing_snps_bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

# Create BED DataFrame for single base
data_to_save = df_allele.drop_duplicates('RSID')
bed_single = pd.DataFrame({
    'chrom': data_to_save ['chr'],
    'chromStart': (data_to_save['pos_hg38'] - 1).astype(int),  # Convert to 0-based
    'chromEnd': data_to_save['pos_hg38'].astype(int),        # 1-based, non-inclusive
    'name': data_to_save['RSID']
})

bed_single.to_csv('mouse_brain_causal_minor_increasing_snps.bed', sep='\t', header=False, index=False)

In [20]:
#HMC3
df_allele = pd.read_csv('/media/zihengc/T7/mpra3_lib_analysis/allele_differences_withoutcontrol/20240817_allele_only_for_plotting/annotated_20240812_comparative_HMC3_alleleOnly.csv',index_col=0)
df_allele['snp_bp_sequence'] = snp_data['Sequence'].tolist()

# Create SeqRecord objects
seq_records = []
data_to_save = df_allele.drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['snp_bp_sequence']):
        record = SeqRecord(
            Seq(row['snp_bp_sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('HMC3_all_snps_bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')


# Create SeqRecord objects
seq_records = []
data_to_save = df_allele[df_allele['fdr']<=0.05].drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['snp_bp_sequence']):
        record = SeqRecord(
            Seq(row['snp_bp_sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('HMC3_sig_snps_bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')


# Create SeqRecord objects
seq_records = []
data_to_save = df_allele[df_allele['fdr']>0.05].drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['snp_bp_sequence']):
        record = SeqRecord(
            Seq(row['snp_bp_sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('HMC3_nonsig_snps_bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')
########################################################################################################################################
# Create BED DataFrame for single base
data_to_save = df_allele.drop_duplicates('RSID')
bed_single = pd.DataFrame({
    'chrom': data_to_save ['chr'],
    'chromStart': data_to_save['pos_hg38'] - 1,  # Convert to 0-based
    'chromEnd': data_to_save['pos_hg38'],        # 1-based, non-inclusive
    'name': data_to_save['RSID']
})


bed_single.to_csv('HMC3_all_snps.bed', sep='\t', header=False, index=False)

# Create BED DataFrame for single base
data_to_save = df_allele[df_allele['fdr']<=0.05].drop_duplicates('RSID')
bed_single = pd.DataFrame({
    'chrom': data_to_save ['chr'],
    'chromStart': data_to_save['pos_hg38'] - 1,  # Convert to 0-based
    'chromEnd': data_to_save['pos_hg38'],        # 1-based, non-inclusive
    'name': data_to_save['RSID']
})


bed_single.to_csv('HMC3_sig_snps.bed', sep='\t', header=False, index=False)


df_allele = pd.read_csv('/media/zihengc/T7/mpra3_lib_analysis/causal_alleles/20240819_HMC3_CausalMinorAllele_ReducingTranscription_combined_withMAD.csv' ,index_col=0).drop('SNP_ID',axis=1)
df_allele = pd.merge(df_allele,snp_data,left_on='RSID',right_on='SNP_ID').drop_duplicates('RSID')

# Create SeqRecord objects
seq_records = []
data_to_save = df_allele[df_allele['fdr']<=0.05].drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['Sequence']):
        record = SeqRecord(
            Seq(row['Sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('HMC3_causal_minor_reducing_snps_bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

# Create BED DataFrame for single base
data_to_save = df_allele.drop_duplicates('RSID')
bed_single = pd.DataFrame({
    'chrom': data_to_save ['chr'],
    'chromStart': (data_to_save['pos_hg38'] - 1).astype(int),  # Convert to 0-based
    'chromEnd': data_to_save['pos_hg38'].astype(int),        # 1-based, non-inclusive
    'name': data_to_save['RSID']
})

bed_single.to_csv('HMC3_causal_minor_reducing_snps.bed', sep='\t', header=False, index=False)



df_allele = pd.read_csv('/media/zihengc/T7/mpra3_lib_analysis/causal_alleles/20240819_HMC3_CausalMinorAllele_IncreasingTranscription_combined_withMAD.csv' ,index_col=0).drop('SNP_ID',axis=1)
df_allele = pd.merge(df_allele,snp_data,left_on='RSID',right_on='SNP_ID').drop_duplicates('RSID')

# Create SeqRecord objects
seq_records = []
data_to_save = df_allele[df_allele['fdr']<=0.05].drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['Sequence']):
        record = SeqRecord(
            Seq(row['Sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('HMC3_causal_minor_reducing_snps_bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

# Create BED DataFrame for single base
data_to_save = df_allele.drop_duplicates('RSID')
bed_single = pd.DataFrame({
    'chrom': data_to_save ['chr'],
    'chromStart': (data_to_save['pos_hg38'] - 1).astype(int),  # Convert to 0-based
    'chromEnd': data_to_save['pos_hg38'].astype(int),        # 1-based, non-inclusive
    'name': data_to_save['RSID']
})

bed_single.to_csv('HMC3_causal_minor_increasing_snps.bed', sep='\t', header=False, index=False)

In [21]:
#HEK293T
df_allele = pd.read_csv('/media/zihengc/T7/mpra3_lib_analysis/allele_differences_withoutcontrol/20240817_allele_only_for_plotting/annotated_20240719_comparative_HEK293T_alleleOnly.csv',index_col=0)
df_allele['snp_bp_sequence'] = snp_data['Sequence'].tolist()

# Create SeqRecord objects
seq_records = []
data_to_save = df_allele.drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['snp_bp_sequence']):
        record = SeqRecord(
            Seq(row['snp_bp_sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('HEK293T_all_snps_bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')


# Create SeqRecord objects
seq_records = []
data_to_save = df_allele[df_allele['fdr']<=0.05].drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['snp_bp_sequence']):
        record = SeqRecord(
            Seq(row['snp_bp_sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('HEK293T_sig_snps_bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')


# Create SeqRecord objects
seq_records = []
data_to_save = df_allele[df_allele['fdr']>0.05].drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['snp_bp_sequence']):
        record = SeqRecord(
            Seq(row['snp_bp_sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('HEK293T_nonsig_snps_bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')
########################################################################################################################################
# Create BED DataFrame for single base
data_to_save = df_allele.drop_duplicates('RSID')
bed_single = pd.DataFrame({
    'chrom': data_to_save ['chr'],
    'chromStart': data_to_save['pos_hg38'] - 1,  # Convert to 0-based
    'chromEnd': data_to_save['pos_hg38'],        # 1-based, non-inclusive
    'name': data_to_save['RSID']
})


bed_single.to_csv('HEK293T_all_snps.bed', sep='\t', header=False, index=False)

# Create BED DataFrame for single base
data_to_save = df_allele[df_allele['fdr']<=0.05].drop_duplicates('RSID')
bed_single = pd.DataFrame({
    'chrom': data_to_save ['chr'],
    'chromStart': data_to_save['pos_hg38'] - 1,  # Convert to 0-based
    'chromEnd': data_to_save['pos_hg38'],        # 1-based, non-inclusive
    'name': data_to_save['RSID']
})


bed_single.to_csv('HEK293T_sig_snps.bed', sep='\t', header=False, index=False)


df_allele = pd.read_csv('/media/zihengc/T7/mpra3_lib_analysis/causal_alleles/20240819_HEK293T_CausalMinorAllele_ReducingTranscription_combined_withMAD.csv' ,index_col=0).drop('SNP_ID',axis=1)
df_allele = pd.merge(df_allele,snp_data,left_on='RSID',right_on='SNP_ID').drop_duplicates('RSID')

# Create SeqRecord objects
seq_records = []
data_to_save = df_allele[df_allele['fdr']<=0.05].drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['Sequence']):
        record = SeqRecord(
            Seq(row['Sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('HEK293T_causal_minor_reducing_snps_bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

# Create BED DataFrame for single base
data_to_save = df_allele.drop_duplicates('RSID')
bed_single = pd.DataFrame({
    'chrom': data_to_save ['chr'],
    'chromStart': (data_to_save['pos_hg38'] - 1).astype(int),  # Convert to 0-based
    'chromEnd': data_to_save['pos_hg38'].astype(int),        # 1-based, non-inclusive
    'name': data_to_save['RSID']
})

bed_single.to_csv('HEK293T_causal_minor_reducing_snps.bed', sep='\t', header=False, index=False)

'/media/zihengc/T7/mpra3_lib_analysis/causal_alleles/20240819_HEK293T_CausalMinorAllele_ReducingTranscription_combined_withMAD.csv'
'/media/zihengc/T7/mpra3_lib_analysis/causal_alleles/20240819_HEK293T_CausalMinorAllele_IncreasingTranscription_combined_withMAD.csv'
df_allele = pd.read_csv('/media/zihengc/T7/mpra3_lib_analysis/causal_alleles/20240819_HEK293T_CausalMinorAllele_IncreasingTranscription_combined_withMAD.csv' ,index_col=0).drop('SNP_ID',axis=1)
df_allele = pd.merge(df_allele,snp_data,left_on='RSID',right_on='SNP_ID').drop_duplicates('RSID')

# Create SeqRecord objects
seq_records = []
data_to_save = df_allele[df_allele['fdr']<=0.05].drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['Sequence']):
        record = SeqRecord(
            Seq(row['Sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('HEK293T_causal_minor_reducing_snps_bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

# Create BED DataFrame for single base
data_to_save = df_allele.drop_duplicates('RSID')
bed_single = pd.DataFrame({
    'chrom': data_to_save ['chr'],
    'chromStart': (data_to_save['pos_hg38'] - 1).astype(int),  # Convert to 0-based
    'chromEnd': data_to_save['pos_hg38'].astype(int),        # 1-based, non-inclusive
    'name': data_to_save['RSID']
})

bed_single.to_csv('HEK293T_causal_minor_increasing_snps.bed', sep='\t', header=False, index=False)

# Clustered SNPs from 6.1

In [22]:
#THP1 Macrophage
df_allele = pd.read_csv('allele_clustering/normalized_snp_mpra_ml_cluster_fixed_20241024.csv',index_col=0)
df_allele['snp_bp_sequence'] = snp_data['Sequence'].tolist()
df_haploreg = pd.read_csv("indexing/SNP_haploregannotatr_ATAC_tss_SNPFunction_spliceai_bayesianML_Bellenguez_202401021.csv",index_col=0)
df_allele =  pd.merge(df_allele[['Cluster','snp_bp_sequence']],df_haploreg[['chr','RSID','pos_hg38']],left_index=True,right_index=True)

In [23]:
# Create SeqRecord objects
seq_records = []
data_to_save = df_allele[df_allele['Cluster']==1].drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['snp_bp_sequence']):
        record = SeqRecord(
            Seq(row['snp_bp_sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)
# Write to a FASTA file
with open('Cluster1_snps_bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

# Create SeqRecord objects
seq_records = []
data_to_save = df_allele[df_allele['Cluster']!=1].drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['snp_bp_sequence']):
        record = SeqRecord(
            Seq(row['snp_bp_sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)
# Write to a FASTA file
with open('Excluding_Cluster1_snps_bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

In [24]:
# Create SeqRecord objects
seq_records = []
data_to_save = df_allele[df_allele['Cluster']==2].drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['snp_bp_sequence']):
        record = SeqRecord(
            Seq(row['snp_bp_sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)
# Write to a FASTA file
with open('Cluster2_snps_bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

# Create SeqRecord objects
seq_records = []
data_to_save = df_allele[df_allele['Cluster']!=2].drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['snp_bp_sequence']):
        record = SeqRecord(
            Seq(row['snp_bp_sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)
# Write to a FASTA file
with open('Excluding_Cluster2_snps_bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

In [25]:
# Create SeqRecord objects
seq_records = []
data_to_save = df_allele[df_allele['Cluster']==3].drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['snp_bp_sequence']):
        record = SeqRecord(
            Seq(row['snp_bp_sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)
# Write to a FASTA file
with open('Cluster3_snps_bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

# Create SeqRecord objects
seq_records = []
data_to_save = df_allele[df_allele['Cluster']!=3].drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['snp_bp_sequence']):
        record = SeqRecord(
            Seq(row['snp_bp_sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)
# Write to a FASTA file
with open('Excluding_Cluster3_snps_bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')